# Plate-solving robustness: detection starvation on SS Leo

A batch run over the **SS Leo** field (2026-04-18) lost several frames to WCS
("plate") solve failures — e.g. `Light_SS Leo_30.0s_IRCUT_20260418-003002.fit` —
even though [astrometry.net](https://nova.astrometry.net) solves the same frames
on upload. This notebook diagnoses why and documents the fix. It is the
reproducible record behind the change for the software paper.

**Finding.** The failure was *not* the detection threshold (the first suspect).
It was two compounding brittlenesses:

1. **Detection starvation.** `eloy.detection.stars_detection` applies a 5×5
   morphological *opening*, so a star must hold a solid 5×5 above-threshold core
   to survive. On these frames a field with **~300 Gaia stars (G ≤ 15)** and
   ~17–26 genuinely detectable stars (photutils) yielded only **6–10**
   detections. Lowering the threshold barely changes this — the *opening*, not
   the threshold, does the culling.
2. **Brittle matcher.** `align` fed the brightest 15 detections against the
   brightest 15 Gaia stars with a tight `tolerance=1`. With only ~10 starved
   detections, that 15/15, tol=1 match is the single failing configuration.

**Fix shipped.** Reduce the detection opening (5 → 3); this alone recovers every
observed frame. The image/Gaia counts were also split into independent constants
(`N_IMAGE_STARS_ALIGN` / `N_GAIA_STARS_ALIGN`, both default 15) and the hardcoded
`tolerance` extracted to `WCS_MATCH_TOLERANCE` (= 1, unchanged) — documented
robustness levers, no behaviour change at the defaults.

> **Reproducibility.** The raw SS Leo FITS frames are not in the repository
> (they live in the author's data store). Set `DATA` below to a local copy to
> re-run. The committed notebook is stored **without cell outputs**; the key
> figure is committed alongside as a PNG.

In [ ]:
import contextlib
import io
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from astropy.visualization import ZScaleInterval
from photutils.detection import DAOStarFinder

from eloy import detection
from bandaid.photometry import (
    THRESH,
    DETECTION_OPENING,
    N_IMAGE_STARS_ALIGN,
    N_GAIA_STARS_ALIGN,
    WCS_MATCH_TOLERANCE,
    calibration_sequence,
)
from bandaid.catalog import cached_gaia_radecs
from twirl import compute_wcs

# Point this at a local copy of the SS Leo raw frames (not shipped in the repo).
DATA = Path(
    "/Users/mattcraig/Dropbox/MSUM/Research/photometry-transform-stuff/"
    "eloy/stwg/SS Leo/SS_Leo_raw_data_and_json"
)

# Frames sampled around the failure (003002/003103/003444 were the failures).
WINDOW = [
    "002626", "002728", "002759", "002900", "002931",
    "003002", "003103", "003134", "003205", "003444",
]
FAILING = ["003002", "003103", "003444"]


def path_for(stamp):
    return sorted(DATA.glob(f"Light_SS Leo_30.0s_IRCUT_2026041*-{stamp}.fit"))[0]

## Gaia reference catalog for the field

Query Gaia exactly as `prepare_batch` does (from the first frame's pointing,
cut at G ≤ 15, brightest first). This is the reference pool twirl matches against.

In [ ]:
_, meta, _, _, _ = calibration_sequence(path_for(WINDOW[0]))
center = (meta["ra"], meta["dec"])
radecs, mags = cached_gaia_radecs(center, 2 * meta["fov_rad"])
keep = mags <= 15
radecs, mags = radecs[keep], mags[keep]
order = np.argsort(mags)  # brightest first
radecs = radecs[order]
print(f"Gaia stars (G<=15) in the field: {len(radecs)}   center={center}")

## 1. Detection starvation

Compare three counts per frame: stars genuinely detectable
(photutils `DAOStarFinder` at a standard 5σ cut), what the pipeline's own
detector returns at the **old** 5×5 opening, and at the **new** 3×3 opening.
The threshold is held at the pipeline's `THRESH` throughout, so any change is the
opening's doing.

In [ ]:
def dao_count(data):
    # photutils baseline: genuinely detectable stars at a standard 5-sigma cut.
    _, median, std = sigma_clipped_stats(data, sigma=3.0)
    sources = DAOStarFinder(fwhm=4.0, threshold=5.0 * std)(data - median)
    return 0 if sources is None else len(sources)


def eloy_count(data, opening):
    return len(detection.stars_detection(data, threshold=THRESH, opening=opening))


print(f"{'frame':>7} {'photutils(5s)':>13} {'eloy open=5':>12} {'eloy open=3':>12}")
print("-" * 48)
for stamp in WINDOW:
    data = fits.getdata(path_for(stamp)).astype(float)
    print(f"{stamp:>7} {dao_count(data):>13} "
          f"{eloy_count(data, 5):>12} {eloy_count(data, 3):>12}")

The old opening (5) returns far fewer stars than the field contains; the new
opening (3) recovers most of them. The figure below shows the failing frame
003002 with its top-15 detections at each opening (same z-scale stretch).

In [ ]:
z = ZScaleInterval()
fig, axes = plt.subplots(1, 2, figsize=(11, 5.4))
data = fits.getdata(path_for("003002")).astype(float)
vmin, vmax = z.get_limits(data)
for ax, opening in zip(axes, (5, 3)):
    regions = detection.stars_detection(data, threshold=THRESH, opening=opening)
    ax.imshow(data, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)
    ys = [r.centroid[0] for r in regions[:15]]
    xs = [r.centroid[1] for r in regions[:15]]
    ax.scatter(xs, ys, s=80, facecolors="none", edgecolors="red", lw=1.2)
    ax.set_title(f"003002  opening={opening}  (N={len(regions)})")
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("SS Leo 003002 - top-15 detections; opening 5 starves, 3 recovers",
             fontsize=13)
fig.tight_layout()
fig.savefig("plate_solving_detection_starvation.png", dpi=110, bbox_inches="tight")
plt.show()

![detection starvation on 003002](plate_solving_detection_starvation.png)

## 2. Lever sweep — what recovers the solve?

Run the *real* `twirl.compute_wcs` on each failing frame across detection
`opening`, the star budget `n` (slicing both lists equally here), and the match
`tolerance`. Status is twirl's outcome: `OK`, `None` (no match), or the exception
it raised.

In [ ]:
def eloy_coords(data, opening):
    regions = detection.stars_detection(data, threshold=THRESH, opening=opening)
    return np.array([(r.centroid[1], r.centroid[0]) for r in regions])


def try_twirl(coords, refs, n, tol):
    if len(coords) < 4:
        return "too_few_det"
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            wcs = compute_wcs(coords[:n], refs[:n], tolerance=tol)
    except Exception as e:  # noqa: BLE001
        return type(e).__name__
    return "OK" if wcs is not None else "None"


for stamp in FAILING:
    data = fits.getdata(path_for(stamp)).astype(float)
    print(f"===== frame {stamp} =====")
    print(f"  {'opening':>7} {'Ndet':>5} | twirl(n, tol) -> status")
    for opening in (5, 3, 2):
        coords = eloy_coords(data, opening)
        cells = [f"({n},{tol})={try_twirl(coords, radecs, n, tol)}"
                 for n in (15, 25) for tol in (1, 3)]
        print(f"  {opening:>7} {len(coords):>5} | " + "  ".join(cells))
    print()

`(15, tol=1)` at `opening=5` is the lone failing combination — exactly the old
defaults. Reducing the opening to 3 makes every combination solve, including the
old `(15, 1)`.

## 3. Image/Gaia asymmetry and its cost

twirl matches asterisms (4-star quads) by shape; a solve needs 4 stars present in
*both* lists. Image detections rank by measured peak while Gaia ranks by catalog
magnitude, so the two "brightest-15" subsets diverge under clouds/colour. Feeding
**more Gaia references than image stars** raises that overlap — which rescues the
*starved* (opening=5) frames — but its cost grows like C(n_gaia, 4). Here image
stars are capped at 15 while the Gaia count varies; times are the median solve.

In [ ]:
def timed_twirl(coords, refs, n_img, n_gaia, tol, repeat=3):
    if len(coords) < 4:
        return "too_few_det", float("nan")
    times, status = [], "?"
    for _ in range(repeat):
        t0 = time.perf_counter()
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                wcs = compute_wcs(coords[:n_img], refs[:n_gaia], tolerance=tol)
            status = "OK" if wcs is not None else "None"
        except Exception as e:  # noqa: BLE001
            status = type(e).__name__
        times.append(time.perf_counter() - t0)
    return status, float(np.median(times))


for opening in (5, 3):
    print(f"########## detection opening={opening} (image stars capped at 15) ####")
    for stamp in FAILING:
        coords = eloy_coords(fits.getdata(path_for(stamp)).astype(float), opening)
        print(f"  frame {stamp}: {len(coords)} detections")
        print(f"    {'n_gaia':>6} {'tol':>4} {'status':>12} {'median_s':>10}")
        for n_gaia in (15, 20, 25, 30):
            for tol in (1, 3):
                status, dt = timed_twirl(coords, radecs, 15, n_gaia, tol)
                print(f"    {n_gaia:>6} {tol:>4} {status:>12} {dt:>10.3f}")
        print()

At `opening=5`, a deeper Gaia pool genuinely rescues the starved frames. At
`opening=3` every frame already solves at Gaia=15, and the time climbs steeply
with the Gaia count (≈ C(n_gaia, 4)). So the opening fix is both sufficient and
cheaper than cranking the Gaia pool — which is why the shipped defaults keep
Gaia=15 and treat the larger pool as a lever, not a default.

## 4. The shipped fix solves every frame

Confirm the failing frames now solve with the values actually shipped in
`bandaid.photometry`.

In [ ]:
print(f"shipped defaults: DETECTION_OPENING={DETECTION_OPENING}, "
      f"N_IMAGE_STARS_ALIGN={N_IMAGE_STARS_ALIGN}, "
      f"N_GAIA_STARS_ALIGN={N_GAIA_STARS_ALIGN}, "
      f"WCS_MATCH_TOLERANCE={WCS_MATCH_TOLERANCE}\n")
for stamp in FAILING:
    coords = eloy_coords(fits.getdata(path_for(stamp)).astype(float),
                         DETECTION_OPENING)
    status = try_twirl(coords, radecs, N_IMAGE_STARS_ALIGN, WCS_MATCH_TOLERANCE)
    # (image and Gaia caps are equal at the defaults, so one budget arg suffices)
    print(f"  {stamp}: {len(coords)} detections -> WCS {status}")